# Cross BC Alignment

__Contributors:__ Curtis Smith, Marissela Gomez (marissela.gomez@stantec.com)  
__Last updated:__ 2/6/2026  

__Notebook Description:__  
- Loads the hdf file results from the downstream HEC-RAS model hdf output file
- gets plan information from SST out files
- overwrites BC conditions (external) as a rating curve to align WSELs at boundary
- ideally a breakline would be added along the BC to facilitate cell alignment (not coded)

__Inputs:__ 
- json files with dictionaries for feature connectivity  
- downstream model with tie in events run for our current model within inputs under "cloud_outputs" folder
- upstream model part of inputs must have auto_bc_creation results in outputs folder

__Outputs:__
- project files for hec-ras, including plan and unsteady flow data for tie in events.

__User QC of output after the notebook is done running:__  
- review that all expected events are in the model as dss files 

## DETAILED STEPS
1) Assume that the post auto bc creation files are included in the "outputs folder"
2) Assume that the cloud outputs from the downstream model are included in the inputs/cloud_output folder
3) Identify upstream hucs
4) Identify pair of overlapping junctions (using dictionary files)
5) Identify storm event for each frequency that controls flooding for each pair of junctions
6) Copy relevant output plan DSS file to upstream huc hydrology folder
7) Assign rating curve to upstream hucs based on the stage/flow relationship for the falling limb of the downstream models inflow hydrograph


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#imports
import os
import pathlib as pl

## Set directories and user defined variables

In [3]:
#set working directory and folder variables
os.chdir('..')
home = pl.Path(os.getcwd())

In [4]:
home = pl.Path(home)
print('home is at ',home)

from src.hdf import *
from src.inputs_py_files.cross_bc_alignment_inputs import *
print('Necessary folders have been created for inputs. Ensure they are placed in the expected location.')

home is at  c:\_code\hms_to_ras_sst
Necessary folders have been created for inputs. Ensure they are placed in the expected location.


In [5]:
#hdf version check
print(h5py.__version__)# Make sure the version is 2.9 or above

3.8.0


In [6]:
#hard variables for project
huc_map = {}
model_name #set within the input .py file. This will be the folder name.

'wy_bh_1008001302'

## Get Relevant RAS Files

In [7]:
#This is finding the HEC-RAS files for the model, which is the downstream of our target. It must be in the cloud_output folder.
plan_path, in_geom_path, in_flow_path = get_current_ras_files(inputs/project/'cloud_output'/huc/f'{model_name}.prj')

## Get Dictionary files

In [8]:
#get upstream models
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_Junctions.json') as src:
    huc_js = json.load(src)
with open(inputs/project/'dictionaries'/'junc_res_sink_next_junc_down.json') as src:
    j_to_j = json.load(src)
with open(inputs/project/'dictionaries'/'event_tie_ins'/'all_model_ds_event_data.json') as src:
    tie_in_events = json.load(src)
with open(inputs/project/'dictionaries'/'event_tie_ins'/'rating_or_stage.json') as rc:
    rc_dict = json.load(rc)

## Get HUC BC connections

In [9]:
us_hucs = []
us_j = []
bc_connections = {'junctions':{},'dss_path':{},'ts':{}}

#If data required to create boundary conditions for all upstream hucs of the target huc is not available, temporarily overwrite us_hucs to only include those with data
if temporary_overwrite == True:
    print('Temporarily overwriting upstream hucs to only those with available data')
    temp_us_hucs = temporary_overwrite_us_huc #list of hucs with available data, defined in the cross_bc_alignment_inputs.py file
    print(temp_us_hucs)
    for temp_us_huc in temp_us_hucs:
        val = huc_connect_huc[temp_us_huc]
        if val == huc:
            us_hucs.append(temp_us_huc)
            us_j.append(huc_connect_j[temp_us_huc])
            bc_connections['junctions'][temp_us_huc] = huc_connect_j[temp_us_huc]
            bc_connections['dss_path'][temp_us_huc] = {}
        else:
            print(f'Issue: {temp_us_huc} not connected to expected downstream huc {huc}')
else:
    for key, val in huc_connect_huc.items():
        if val == huc:
            us_hucs.append(key)
            us_j.append(huc_connect_j[key])
            bc_connections['junctions'][key] = huc_connect_j[key]
            bc_connections['dss_path'][key] = {}

Temporarily overwriting upstream hucs to only those with available data
['1008001301']


In [10]:
#this print statement should include your target huc
print(f'Creating new ds boundary conditions for {us_hucs}')

Creating new ds boundary conditions for ['1008001301']


## Copy DSS Files

In [11]:
bc_connections['junctions'].items()

dict_items([('1008001301', 'HUC_013_J_33')])

In [12]:
#outputs from auto_bc_creation must be present in the outputs folder before running this section. This is needed for each us_huc listed.
for us_huc, j in  bc_connections['junctions'].items():
    #copy autobc creation files to eb_mod folder for adjustment
    assert os.path.exists(str(outputs_base/project/f'{model_prefix}{us_huc}')), 'Please run auto bc creation tool before running cross bc alignment tool'
    if os.path.exists(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}')):
        shutil.rmtree(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'))
    shutil.copytree(str(outputs_base/project/f'{model_prefix}{us_huc}'),str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'))
    
    #create output folder if it doesn't exist
    if not os.path.exists(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'):
        os.makedirs(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology')
    plan_dss_in = str(inputs/project/'cloud_output'/huc/f'{model_name}.dss')
    plan_dss_out = str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'/f'{model_name}.dss')
    shutil.copy(plan_dss_in,plan_dss_out)

In [13]:
bc_connections

{'junctions': {'1008001301': 'HUC_013_J_33'},
 'dss_path': {'1008001301': {}},
 'ts': {}}

# Review and update Upstream HUCs

## Update DSS Files

In [14]:
## get stage hydrograph from us inflow junction and convert to a rating curve and apply it to us_huc downstream BC output file
flow_number_dict = {}
for us_huc, j in  bc_connections['junctions'].items():
    rating_or_stage = rc_dict[us_huc]
    ds_huc = huc
    #get downstream bc time series and dss file
    dss_ds = str(inputs/project/'cloud_output'/f'{ds_huc}'/f'{model_name}.dss')
    fid_ds = HecDss.Open(str(dss_ds))
    pathname_pattern ="/*/*/*/*/*/*/"
    dss_list_ds = fid_ds.getPathnameList(pathname_pattern,sort=1)
    
    #get upstream dss file and plan number
    us_model_name = f'{model_prefix}{us_huc}' #folder within inputs folder
    ##read plan names and get relevant plan output file to be copied to upstream hucs
    plan_number_dict = {}
    flow_number_dict[us_huc] ={}
    #get plan and flow path info
    plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/f'{us_model_name}.prj')
    plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
    #for p in plan_files:
    for p in plan_files[2:3]:
        #read old text file
        with open(p, "r+") as f:
            file_contents = f.read()
        plan_title_finder= '(?<=Plan Title=)[\d\w]+(?=_output)'
        f = re.search(plan_title_finder,file_contents)
        flow_finder = '(?<=Plan Title=)[\d\w]+(?=_output)'
        u = re.search(flow_finder,file_contents)
        if f:
            f_num = re.search('.p[\d]+',p)
            plan_number_dict[f.group()] = f_num.group()
            u_num = re.search('u[\d]+',file_contents)
            flow_number_dict[us_huc][f.group()] = u_num.group()
    j_ = list(tie_in_events[us_huc].keys())        
    for ri, category in tie_in_events[us_huc][j_[0]].items():
        event_us = category['model_tie_in_event']
        event_us_flow_expected = category['model_tie_in_flow']
        event_name_us = event_us.replace('_R','_M').replace('-','_')
        event_finder_us = f'[\d\w\S\s]+{event_name_us[:event_name_us.find("_Y")]}_[\d\s\S]+{event_name_us[event_name_us.find("_Y"):]}[\d\w\S]+'
        #get all relevant dss paths and condense using wildcard
        
        r_us = re.compile(event_finder_us)
        dss_list_us = glob.glob(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'/'*.dss'))
        dss_matches_us = list(filter(r_us.match, dss_list_us))
        
        #get downstream bc time series and dss file
        dss_us = str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'/f'{dss_matches_us[0]}')
        assert len(dss_matches_us)>0, f'no output data found for {event_us} in {dss_us}'
        fid_us = HecDss.Open(str(dss_us))
        pathname_pattern ="/*/*/*/*/*/*/"
        dss_list_us = fid_us.getPathnameList(pathname_pattern,sort=1)

        
        event_ds = category['ds_tie_in_event']
        event_name_ds = event_ds.replace('_R','_M').replace('-','_')
        # f'\S+(R\d+-)?{evnt_name_u[:evnt_name_u.find("_Y")]}_[\d\s\S]+{evnt_name_u[evnt_name_u.find("_Y"):]}_output\.dss'

        event_finder_ds = f'[\d\w\S\s]+{j}/[\d\w\S]+{event_name_ds[:event_name_ds.find("_Y")]}_[\d\s\S]+{event_name_ds[event_name_ds.find("_Y"):]}_output_[\d\w\S]+'
        #get all relevant dss paths and condense using wildcard
        r_ds = re.compile(event_finder_ds)
        dss_matches_ds = list(filter(r_ds.match, dss_list_ds))

        # print(len(dss_matches_ds),dss_matches_ds)

        stage_data_ds = list(filter(re.compile('[\d\w\S\s]+/STAGE/[\d\w\S\s]+').match, dss_matches_ds))
        flow_data_ds = list(filter(re.compile('[\d\w\S\s]+/FLOW/[\d\w\S\s]+').match, dss_matches_ds))
        print(stage_data_ds,flow_data_ds)
        
        event_finder_us_w = f'[\d\W\w\S\s]+/{j}/FLOW-COMBINE/[\d\W\w\S\s]+'
        r_us = re.compile(event_finder_us_w)
        flow_data_us = list(filter(r_us.match, dss_list_us))
        print(len(flow_data_us),flow_data_us)
        
        #write new dss file
        hg_name = create_bc_from_junc(bc_connections,fid_ds,fid_us,stage_data_ds,flow_data_ds,flow_data_us,rating_or_stage,event_us_flow_expected)
        bc_connections['dss_path'][us_huc][ri] = hg_name
        

['/BCLINE/M SF Shoshone R: HUC_013_J_33/STAGE/01May1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/', '/BCLINE/M SF Shoshone R: HUC_013_J_33/STAGE/01Jun1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/', '/BCLINE/M SF Shoshone R: HUC_013_J_33/STAGE/01Jul1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/'] ['/BCLINE/M SF Shoshone R: HUC_013_J_33/FLOW/01May1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/', '/BCLINE/M SF Shoshone R: HUC_013_J_33/FLOW/01Jun1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/', '/BCLINE/M SF Shoshone R: HUC_013_J_33/FLOW/01Jul1980/1Hour/P07_M_HUC10080013_Y202_E0002_output_53/']
3 ['//HUC_013_J_33/FLOW-COMBINE/01May2022/15Minute/RUN:P01_M-HUC10080013-Y465-E0002/', '//HUC_013_J_33/FLOW-COMBINE/01Jun2022/15Minute/RUN:P01_M-HUC10080013-Y465-E0002/', '//HUC_013_J_33/FLOW-COMBINE/01Jul2022/15Minute/RUN:P01_M-HUC10080013-Y465-E0002/']
max stage is 6441.4580078125
max flow associated with max stage is 7940.08056640625
[[6439.0767 6440.958  6441.458  6442.458 ]]


In [15]:
flow_number_dict = {}
domain_name_dict = {}
for us_huc, j in  bc_connections['junctions'].items():
    us_model_name = f'{model_prefix}{us_huc}' #folder within inputs folder for the target huc
    ##read plan names and get relevant plan output file to be copied to upstream hucs
    plan_number_dict = {}
    flow_number_dict[us_huc] ={}
    plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/f'{us_model_name}.prj')
    
    #geo hdf
    hf_geo = h5py.File(str(in_geom_path)+'.hdf','r')
    #get domain names
    domain_geo = str(list(hf_geo['Geometry']['2D Flow Areas']['Attributes'])[0][0]).strip("'b\'")
    domain_name_dict[us_huc] = domain_geo
    plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
    for p in plan_files:
        #read old text
        with open(p, "r+") as f:
            file_contents = f.read()
        plan_title_finder= '(?<=Plan Title=)[\d\w]+(?=_output)'
        f = re.search(plan_title_finder,file_contents)
        flow_finder = '(?<=Plan Title=)[\d\w]+(?=_output)'
        u = re.search(flow_finder,file_contents)
        if f:
            f_num = re.search('.p[\d]+',p)
            plan_number_dict[f.group()] = f_num.group()
            u_num = re.search('u[\d]+',file_contents)
            flow_number_dict[us_huc][f.group()] = u_num.group()

In [16]:
for us_huc in  bc_connections['dss_path'].keys():
    flow_files = glob.glob(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'*.u*[!.hdf]'))
    junction = bc_connections['junctions'][us_huc]
    domain_name = domain_name_dict[us_huc]
    for ri,dss_path in bc_connections['dss_path'][us_huc].items():
        
        #########
        j_ = list(tie_in_events[us_huc].keys())
        category = tie_in_events[us_huc][j_[0]][ri]
        event_us = category['model_tie_in_event']   
        event_name_us = event_us.replace('_R','_M').replace('-','_')
        event_finder_us = f'[\d\w\S\s]+{event_name_us[:event_name_us.find("_Y")]}_[\d\s\S]+{event_name_us[event_name_us.find("_Y"):]}[\d\w\S]+'
        #get all relevant dss paths and condense using wildcard
        r_us = re.compile(event_finder_us)
        dss_list_us = glob.glob(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'/'*.dss'))
        dss_matches_us = list(filter(r_us.match, dss_list_us))
        
        #get downstream bc time series and dss file
        dss_us = str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/'Hydrology'/f'{dss_matches_us[0]}')
        current_event = os.path.basename(dss_us).replace('_output.dss','')
        flow_num = flow_number_dict[us_huc][current_event]

        in_flow_path = glob.glob(str(outputs_base/project/'eb_mod'/f'{model_prefix}{us_huc}'/f'*.{flow_num}'))[0]
        dss_path = bc_connections['dss_path'][us_huc][ri]
        dss_name = str(pl.Path(dss_us).stem)+'.dss' 
        if dss_path.split('/')[2][-3:] == '_rc':
            print(dss_path)
            print(in_flow_path)
            #assume rating curve condition
            a = update_flow_file(dss_name,dss_path,in_flow_path,huc,domain_name)
        elif dss_path.split('/')[2][-10:] == '_stage_hyd':
            #assumes ponding condition
            a = update_flow_file_stage(dss_name,dss_path,in_flow_path,huc,domain_name)
        else:
            #doesn't match any conditions
            print('issue with code assumptions, Talk to Curtis or Marissela')
            break


/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P07_M_HUC10080013_Y202_E0002_output_53/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\wy_bh_1008001301.u52
/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P03_M_HUC10080013_Y298_E0002_output_56/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\wy_bh_1008001301.u57
/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P01_M_HUC10080013_Y159_E0001_output_63/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\wy_bh_1008001301.u52
/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P03_M_HUC10080013_Y392_E0002_output_68/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\wy_bh_1008001301.u66
/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P07_M_HUC10080013_Y187_E0002_output_71/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\wy_bh_1008001301.u69
/BCLINE/M SF Shoshone R: HUC_013_J_33_rc/-///P05_M_HUC10080013_Y426_E0002_output_77/
c:\_code\hms_to_ras_sst\outputs\wy_fy23\eb_mod\wy_bh_1008001301\w

## END